# Trisha-v.3.0 (224px)
4 model (EfficientFormer-L1, Swin-Tiny, ConvNeXt-Tiny, CLIP ViT-B/32).
- **Stage 1:** Recyclable + Organic (224px)
- **Stage 2:** Organic + Electronic (224px)
- **Phase 3:** Full 3 classes fine-tune (low LR)
- **Data:** train/ → train_clean/ (rename + dedup only)

In [12]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm

%matplotlib inline

_cwd = Path.cwd()
if (_cwd / 'modules').exists():
    sys.path.insert(0, str(_cwd))
elif (_cwd.parent / 'modules').exists():
    sys.path.insert(0, str(_cwd.parent))
else:
    sys.path.insert(0, str(_cwd.parent.parent))

In [2]:
try:
    import open_clip
    print('open-clip-torch already installed')
except ImportError:
    print('open-clip-torch not available - CLIP model will be skipped')
    open_clip = None

d:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


open-clip-torch already installed


In [13]:
from modules.utils.config import CLASS_LABELS, IMG_SIZE, MEAN, NUM_CLASSES, PROJECT_ROOT, RESULTS, SEED, STD
from modules.models.factory import TrashClassifier
from modules.models.clip_adapter import CLIPAdapter
from modules.training.loss import ClassBalancedLoss
from modules.training.train import fit
from modules.training.collator import MixUpCollator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = RESULTS / 'trisha_v3.0'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Output dir: {OUT_DIR}')
print(f'Project root: {PROJECT_ROOT}')

Device: cuda
Output dir: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\experiments\results\trisha_v3.0
Project root: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble


---
## 1. Setup Data: train_clean/
Copy `train/` →   `train_clean/`, hapus 62 duplicate, rename 3 long-path.

In [4]:
SRC_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train'
TRAIN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train_clean'

if not SRC_DIR.exists():
    raise FileNotFoundError('train/ not found. Download from drive first.')

if TRAIN_DIR.exists():
    print(f'{TRAIN_DIR} already exists, skipping copy.')
else:
    print('Copying train/ -> train_clean/...')
    ret = os.system(f'robocopy "{SRC_DIR}" "{TRAIN_DIR}" /E /COPY:DAT /R:2 /W:2 /NDL /NFL /NJH /NJS')
    print('Done')

Copying train/ -> train_clean/...


Done


In [5]:
long_names = [
    ("1.tumpukan-e-limbah-yang-kacau-dari-laptop-dan-suku-cadang-komputer-yang-dibuang-representasi-visual-yang-luar-biasa-dari-masalah-limbah-elektronik-yang-berkembang-dan-kebutuhan-akan-solusi-daur-ulang-berkelanjutan_73523-11403.jpg", "E_001.jpg"),
    ("12.pile-discarded-motherboards-cpus-cables-disc-drives-hijacked-hardware-heap-concept-electronic-waste-tech-recycling-hardware-reuse-sustainable-technology-environmental-impact_864588-263287.jpg", "E_012.jpg"),
    ("64.dampak-lingkungan-dari-ewaste-tangkap-konsekuensi-lingkungan-dari-pembuangan-ewaste-yang-tidak-tepat-seperti-perangkat-elektronik-yang-mengeluarkan-bahan-kimia-beracun-ke-dalam-tanah-dan-saluran-air-di-lokasi-pembuangan-ilegal_997534-43151.jpg", "E_064.jpg"),
]
def _safe_path(p):
    p = str(p)
    if len(p) > 240 and not p.startswith('\\\\?\\'):
        return '\\\\?\\' + p
    return p

for old_name, new_name in long_names:
    old = TRAIN_DIR / '1_Electronic' / old_name
    new = old.with_name(new_name)
    try:
        os.rename(_safe_path(old), _safe_path(new))
        print(f'{old_name[:40]}... -> {new_name}')
    except FileNotFoundError:
        pass
    except FileExistsError:
        pass
print('Rename done.')

# Scan MD5 + remove duplicates (keep 1 per group)
hash_map = defaultdict(list)
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = TRAIN_DIR / cls
    if not cls_path.is_dir():
        continue
    for fpath in cls_path.iterdir():
        try:
            with open(_safe_path(fpath), 'rb') as f:
                h = hashlib.md5(f.read()).hexdigest()
            hash_map[h].append({'cls': cls, 'fname': fpath.name, 'path': fpath})
        except:
            pass

dup_groups = {h: files for h, files in hash_map.items() if len(files) > 1}
print(f'Duplicate groups: {len(dup_groups)}')

removed = 0
for h, files in dup_groups.items():
    for f in files[1:]:
        os.remove(f['path'])
        removed += 1
print(f'Removed {removed} duplicate files.')

1.tumpukan-e-limbah-yang-kacau-dari-lapt... -> E_001.jpg
12.pile-discarded-motherboards-cpus-cabl... -> E_012.jpg
64.dampak-lingkungan-dari-ewaste-tangkap... -> E_064.jpg
Rename done.
Duplicate groups: 62
Removed 62 duplicate files.


---
## 2. Load & Split Data (train_clean/)

In [6]:
TRAIN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train_clean'
records = []
for label_dir in sorted(TRAIN_DIR.iterdir()):
    if not label_dir.is_dir():
        continue
    for img in sorted(label_dir.glob('*')):
        if img.is_file():
            records.append({'path': str(img.relative_to(PROJECT_ROOT)), 'label': label_dir.name})
df = pd.DataFrame(records)
print(f'Total: {len(df)}')
for cls, count in sorted(Counter(df['label']).items()):
    print(f'  {cls}: {count}')

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f'Train: {len(train_df)}, Val: {len(val_df)}')

Total: 26465
  0_Recyclable: 9990
  1_Electronic: 3931
  2_Organic: 12544
Train: 21172, Val: 5293


---
## 3. Dataset Class + Transforms

In [7]:
_LABEL_TO_IDX = {name: i for i, name in enumerate(CLASS_LABELS)}

def _open_image(path):
    safe = _safe_path(path)
    with open(safe, 'rb') as f:
        return Image.open(BytesIO(f.read())).convert('RGB')

class TrashDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = (PROJECT_ROOT / df['path']).values
        self.labels = df['label'].map(_LABEL_TO_IDX).values
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = _open_image(self.paths[idx])
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

def make_transform(img_size, is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.3, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
            transforms.RandomErasing(p=0.3),
        ])
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

def make_loader(df, img_size, batch_size=32, shuffle=True, use_mixup=False):
    ds = TrashDataset(df, transform=make_transform(img_size, is_train=shuffle))
    collator = MixUpCollator(alpha=0.2, num_classes=3) if (use_mixup and shuffle) else None
    sampler = None
    if shuffle:
        labels = df['label'].map(_LABEL_TO_IDX).values
        counts = torch.bincount(torch.tensor(labels))
        weights = 1.0 / counts[labels].float()
        sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights), replacement=True)
    return DataLoader(
        ds, batch_size=batch_size,
        sampler=sampler, shuffle=False if sampler else shuffle,
        num_workers=0, pin_memory=False,
        collate_fn=collator,
    )

---
## 4. Dataloaders
Stage 1: Recyclable+Organic (224px)
Stage 2: Organic+Electronic (224px)
Final eval: full 3 kelas (224px)

In [8]:
BATCH_SIZE = 32

# Stage 1: Recyclable + Organic
mask_s1 = train_df['label'].isin(['0_Recyclable', '2_Organic'])
train_s1 = train_df[mask_s1].reset_index(drop=True)
val_s1 = val_df[val_df['label'].isin(['0_Recyclable', '2_Organic'])].reset_index(drop=True)
train_loader_s1 = make_loader(train_s1, img_size=224, batch_size=BATCH_SIZE, use_mixup=True)
val_loader_s1 = make_loader(val_s1, img_size=224, shuffle=False)
print(f'Stage 1 train: {len(train_s1)}, val: {len(val_s1)}')

# Stage 2: Organic + Electronic
mask_s2 = train_df['label'].isin(['1_Electronic', '2_Organic'])
train_s2 = train_df[mask_s2].reset_index(drop=True)
val_s2 = val_df[val_df['label'].isin(['1_Electronic', '2_Organic'])].reset_index(drop=True)
train_loader_s2 = make_loader(train_s2, img_size=224, batch_size=BATCH_SIZE, use_mixup=True)
val_loader_s2 = make_loader(val_s2, img_size=224, shuffle=False)
print(f'Stage 2 train: {len(train_s2)}, val: {len(val_s2)}')

# Final full evaluation
train_loader_full = make_loader(train_df, img_size=224, use_mixup=False)
val_loader_full = make_loader(val_df, img_size=224, shuffle=False)
print(f'Full train: {len(train_df)}, val: {len(val_df)}')

C:\Users\Acer\AppData\Local\Temp\ipykernel_10252\2465191917.py:45: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  weights = 1.0 / counts[labels].float()


Stage 1 train: 18027, val: 4507
Stage 2 train: 13180, val: 3295
Full train: 21172, val: 5293


---
## 5. Helper: move .pt to output dir

In [9]:
def move_to_outdir(name):
    for ext in ['.pt', '.json']:
        src = RESULTS / f'{name}{ext}'
        if src.exists():
            shutil.move(str(src), str(OUT_DIR / f'{name}{ext}'))

def eval_model(model, loader):
    model.to(DEVICE).eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc='Eval', leave=False):
            out = model(x.to(DEVICE))
            preds.extend(out.argmax(dim=1).cpu().numpy().tolist())
            labels.extend(y.cpu().numpy().tolist())
    from sklearn.metrics import f1_score, precision_score, recall_score
    f1 = f1_score(labels, preds, average='macro')
    f1_pc = f1_score(labels, preds, average=None, labels=list(range(3))).tolist()
    prec_pc = precision_score(labels, preds, average=None, labels=list(range(3))).tolist()
    rec_pc = recall_score(labels, preds, average=None, labels=list(range(3))).tolist()
    return {'f1_macro': f1, 'f1_per_class': f1_pc, 'precision': prec_pc, 'recall': rec_pc, 'preds': preds, 'labels': labels}

---
## 6. TRAIN: EfficientFormer-L1

In [10]:
MODEL = 'efficientformer_l1'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Stage 1: Recyclable + Organic, 224px
print(f'\nStage 1: Recyclable + Organic @ 224px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=20,
    lr_head=5e-4, lr_finetune=5e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 224px
print(f'\nStage 2: Organic + Electronic @ 224px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=6, epochs_finetune=15,
    lr_head=3e-4, lr_finetune=2e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Phase 3 — Recalibrate (fine-tune all, low LR)
print(f'\n=== {MODEL}: Phase 3 — Recalibrate All @ 224px (full 3 classes, LR=5e-6) ===')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location='cpu'))
fit(model, train_loader_full, val_loader_full,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=8,
    lr_head=0, lr_finetune=5e-6, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

=== efficientformer_l1 ===


Parameters: 11,623,355

Stage 1: Recyclable + Organic @ 224px

=== efficientformer_l1_s1: Phase 1 — Head Only ===


  E01: train_loss=0.4125  val_f1=0.9339  best=0.9339


  E02: train_loss=0.3612  val_f1=0.9360  best=0.9360


  E03: train_loss=0.3481  val_f1=0.9454  best=0.9454


...

  E18: train_loss=0.1498  val_f1=0.9925  best=0.9934


  E19: train_loss=0.1718  val_f1=0.9917  best=0.9934


  E20: train_loss=0.1652  val_f1=0.9913  best=0.9934
  Early stopping at epoch 20



=== efficientformer_l1: Phase 3 — Recalibrate All @ 224px (full 3 classes, LR=5e-6) ===

=== efficientformer_l1_final: Phase 1 — Head Only ===

=== efficientformer_l1_final: Phase 2 — Fine-tune All ===


TypeError: Expected state_dict to be dict-like, got <class 'NoneType'>.

In [15]:
import importlib
import modules.training.train
importlib.reload(modules.training.train)
from modules.training.train import fit

In [16]:
# Phase 3 — Recalibrate (fine-tune all, low LR)
print(f'\n=== {MODEL}: Phase 3 — Recalibrate All @ 224px (full 3 classes, LR=5e-6) ===')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location='cpu'))
fit(model, train_loader_full, val_loader_full,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=8,
    lr_head=0, lr_finetune=5e-6, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')


=== efficientformer_l1: Phase 3 — Recalibrate All @ 224px (full 3 classes, LR=5e-6) ===

=== efficientformer_l1_final: Phase 1 — Head Only ===

=== efficientformer_l1_final: Phase 2 — Fine-tune All ===


  E01: train_loss=1.0943  val_f1=0.9283  best=0.9283


  E02: train_loss=0.2382  val_f1=0.9503  best=0.9503


  E03: train_loss=0.1956  val_f1=0.9567  best=0.9567


  E04: train_loss=0.1626  val_f1=0.9551  best=0.9567


  E05: train_loss=0.1584  val_f1=0.9585  best=0.9585


  E06: train_loss=0.1490  val_f1=0.9578  best=0.9585


  E07: train_loss=0.1537  val_f1=0.9573  best=0.9585


  E08: train_loss=0.1475  val_f1=0.9579  best=0.9585


F1 macro: 0.9578
              precision    recall  f1-score   support

0_Recyclable     0.9490    0.9414    0.9452      1998
1_Electronic     0.9426    0.9822    0.9620       786
   2_Organic     0.9695    0.9629    0.9662      2509

    accuracy                         0.9577      5293
   macro avg     0.9537    0.9622    0.9578      5293
weighted avg     0.9578    0.9577    0.9577      5293

Saved: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\experiments\results\trisha_v3.0\efficientformer_l1.pt/.json


---
## 7. TRAIN: ConvNeXt-Tiny

In [17]:
MODEL = 'convnext_tiny'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Stage 1: Recyclable + Organic, 224px
print(f'\nStage 1: Recyclable + Organic @ 224px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=20,
    lr_head=5e-4, lr_finetune=5e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 224px
print(f'\nStage 2: Organic + Electronic @ 224px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=6, epochs_finetune=15,
    lr_head=3e-4, lr_finetune=2e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Phase 3 — Recalibrate (fine-tune all, low LR)
print(f'\n=== {MODEL}: Phase 3 — Recalibrate All @ 224px (full 3 classes, LR=5e-6) ===')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location='cpu'))
fit(model, train_loader_full, val_loader_full,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=8,
    lr_head=0, lr_finetune=5e-6, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

=== convnext_tiny ===


Parameters: 28,215,395

Stage 1: Recyclable + Organic @ 224px

=== convnext_tiny_s1: Phase 1 — Head Only ===


  E01: train_loss=0.3293  val_f1=0.9631  best=0.9631


  E02: train_loss=0.2650  val_f1=0.9636  best=0.9636


  E03: train_loss=0.2459  val_f1=0.9648  best=0.9648


...

  E05: train_loss=0.0465  val_f1=0.9741  best=0.9743


  E06: train_loss=0.0447  val_f1=0.9746  best=0.9746


  E07: train_loss=0.0455  val_f1=0.9750  best=0.9750


  E08: train_loss=0.0416  val_f1=0.9747  best=0.9750


F1 macro: 0.9745
              precision    recall  f1-score   support

0_Recyclable     0.9598    0.9675    0.9636      1998
1_Electronic     0.9860    0.9860    0.9860       786
   2_Organic     0.9771    0.9709    0.9740      2509

    accuracy                         0.9718      5293
   macro avg     0.9743    0.9748    0.9745      5293
weighted avg     0.9719    0.9718    0.9719      5293

Saved: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\experiments\results\trisha_v3.0\convnext_tiny.pt/.json


---
## 8. TRAIN: CLIP ViT-B/32 (VLM)

In [18]:
# Wrap CLIPAdapter to match fit() interface
class CLIPTrainable(CLIPAdapter):
    @property
    def classifier(self):
        return self.visual_projection
    @property
    def encoder(self):
        return self.clip.visual
    def freeze_encoder(self):
        for p in self.clip.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.clip.visual.parameters():
            p.requires_grad = True

MODEL = 'clip_vit_b32'
print(f'=== {MODEL} ===')

model = CLIPTrainable(clip_variant='ViT-B/32', num_classes=3, device=DEVICE)
clip_params = sum(p.numel() for p in model.clip.visual.parameters() if p.requires_grad)
proj_params = sum(p.numel() for p in model.visual_projection.parameters())
print(f'CLIP encoder: {clip_params:,}, Projection: {proj_params:,}')

# Zero-shot benchmark first
model.to(DEVICE).eval()
prompts = ['a photo of recyclable waste', 'a photo of electronic waste', 'a photo of organic waste']
zs_preds, zs_labels = [], []
for x, y in tqdm(val_loader_full, desc='Zero-shot', leave=False):
    logits = model.zero_shot_classify(x.to(DEVICE), prompts)
    zs_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    zs_labels.extend(y.numpy().tolist())
from sklearn.metrics import f1_score
print(f'Zero-shot F1 macro: {f1_score(zs_labels, zs_preds, average="macro"):.4f}')

# Stage 1: Recyclable + Organic, 224px (fine-tune projection only)
print(f'\nStage 1: Recyclable + Organic @ 224px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=4, epochs_head=10, epochs_finetune=15,
    lr_head=1e-3, lr_finetune=1e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 224px
print(f'\nStage 2: Organic + Electronic @ 224px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=4, epochs_head=6, epochs_finetune=10,
    lr_head=5e-4, lr_finetune=5e-6, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Phase 3 - Recalibrate (fine-tune all, low LR)
print(f'\n=== {MODEL}: Phase 3 - Recalibrate All @ 224px (full 3 classes, LR=1e-6) ===')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location='cpu'))
fit(model, train_loader_full, val_loader_full,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=4, epochs_head=0, epochs_finetune=8,
    lr_head=0, lr_finetune=1e-6, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

=== clip_vit_b32 ===


d:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--laion--CLIP-ViT-B-32-laion2B-s34B-b79K. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


CLIP encoder: 87,849,216, Projection: 1,539


Zero-shot F1 macro: 0.8465

Stage 1: Recyclable + Organic @ 224px

=== clip_vit_b32_s1: Phase 1 — Head Only ===


  E01: train_loss=0.8714  val_f1=0.9227  best=0.9227


...

  E05: train_loss=0.1041  val_f1=0.9747  best=0.9747


  E06: train_loss=0.1042  val_f1=0.9753  best=0.9753


  E07: train_loss=0.1014  val_f1=0.9757  best=0.9757


  E08: train_loss=0.0939  val_f1=0.9763  best=0.9763


F1 macro: 0.9763
              precision    recall  f1-score   support

0_Recyclable     0.9674    0.9665    0.9670      1998
1_Electronic     0.9810    0.9873    0.9841       786
   2_Organic     0.9785    0.9773    0.9779      2509

    accuracy                         0.9747      5293
   macro avg     0.9756    0.9770    0.9763      5293
weighted avg     0.9747    0.9747    0.9747      5293

Saved: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\experiments\results\trisha_v3.0\clip_vit_b32.pt/.json


---
## 8.1 TRAIN: Swin-Tiny

In [19]:
MODEL = 'swin_tiny_patch4_window7_224'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Stage 1: Recyclable + Organic, 224px
print(f'\nStage 1: Recyclable + Organic @ 224px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=20,
    lr_head=5e-4, lr_finetune=5e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 224px
print(f'\nStage 2: Organic + Electronic @ 224px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=6, epochs_finetune=15,
    lr_head=3e-4, lr_finetune=2e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Phase 3 - Recalibrate (fine-tune all, low LR)
print(f'\n=== {MODEL}: Phase 3 - Recalibrate All @ 224px (full 3 classes, LR=5e-6) ===')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location='cpu'))
fit(model, train_loader_full, val_loader_full,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=8,
    lr_head=0, lr_finetune=5e-6, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

=== swin_tiny_patch4_window7_224 ===


d:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--timm--swin_tiny_patch4_window7_224.ms_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Parameters: 27,914,621

Stage 1: Recyclable + Organic @ 224px

=== swin_tiny_patch4_window7_224_s1: Phase 1 — Head Only ===


  E01: train_loss=0.3502  val_f1=0.9490  best=0.9490


  E02: train_loss=0.3039  val_f1=0.9526  best=0.9526


...

  E05: train_loss=0.0911  val_f1=0.9679  best=0.9679


  E06: train_loss=0.0888  val_f1=0.9689  best=0.9689


  E07: train_loss=0.0861  val_f1=0.9688  best=0.9689


  E08: train_loss=0.0889  val_f1=0.9686  best=0.9689


F1 macro: 0.9686
              precision    recall  f1-score   support

0_Recyclable     0.9623    0.9575    0.9599      1998
1_Electronic     0.9546    0.9898    0.9719       786
   2_Organic     0.9779    0.9705    0.9742      2509

    accuracy                         0.9684      5293
   macro avg     0.9649    0.9726    0.9686      5293
weighted avg     0.9685    0.9684    0.9684      5293

Saved: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\experiments\results\trisha_v3.0\swin_tiny_patch4_window7_224.pt/.json


---
## 9. Summary Table

In [20]:
models = ['efficientformer_l1', 'convnext_tiny', 'swin_tiny_patch4_window7_224', 'clip_vit_b32']
rows = []
for m in models:
    path = OUT_DIR / f'{m}.json'
    if path.exists():
        d = json.load(open(path))
        rows.append({
            'model': m,
            'params': f'{sum(p.numel() for p in TrashClassifier(encoder_name=m, num_classes=3).parameters()):,}' if m != 'clip_vit_b32' else '150M+',
            'f1_macro': d['f1_macro'],
            'f1_0_recyclable': d['f1_per_class'][0],
            'f1_1_electronic': d['f1_per_class'][1],
            'f1_2_organic': d['f1_per_class'][2],
            'precision': d['precision'],
            'recall': d['recall'],
        })

df_result = pd.DataFrame(rows).sort_values('f1_macro', ascending=False)
print(df_result.to_string(index=False))

df_result.to_csv(OUT_DIR / 'summary.csv', index=False)
print(f'\nSaved: {OUT_DIR / "summary.csv"}')

                       model     params  f1_macro  f1_0_recyclable  f1_1_electronic  f1_2_organic                                                    precision                                                       recall
                clip_vit_b32      150M+  0.976321         0.966950         0.984147      0.977866  [0.967434869739479, 0.9810366624525917, 0.9784517158818835]  [0.9664664664664665, 0.9872773536895675, 0.977281785571941]
               convnext_tiny 28,215,395  0.974542         0.963609         0.986005      0.974010 [0.9597815292949354, 0.9860050890585241, 0.9771359807460891] [0.9674674674674675, 0.9860050890585241, 0.9709047429254684]
swin_tiny_patch4_window7_224 27,914,621  0.968649         0.959860         0.971893      0.974195 [0.9622736418511066, 0.9546012269938651, 0.9779116465863453]  [0.9574574574574575, 0.989821882951654, 0.9705061777600638]
          efficientformer_l1 11,623,355  0.957809         0.945226         0.961994      0.966207 [0.9490413723511605, 0